In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
from transformers import AutoProcessor, AutoModelForZeroShotImageClassification
import torch
from PIL import Image

# ======================
# Step 1: Load Model
# ======================
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained("chanwkim/monet")
model = AutoModelForZeroShotImageClassification.from_pretrained("chanwkim/monet")
model.to(device)
model.eval()

# ======================
# Step 2: Load Image
# ======================
image_path = "/content/pexels-anna-nekrashevich-6475987.jpg"  # Replace with your image path
image = Image.open(image_path).convert("RGB")
image.show()

# ======================
# Step 3: Define Concepts (Symptoms)
# ======================
concepts = [
    "erythematous",
    "scaly",
    "hyperpigmented",
    "hypopigmented",
    "papular",
    "nodular",
    "vesicular",
    "ulcerated",
    "crusted",
    "acne",
    "eczema",
    "psoriasis",
    "rosacea",
    "skin cancer",
    "fungal infection",
    "skin enzyme"  # ✅ Include this so it's detected
]

# ======================
# Step 4: Run Inference
# ======================
inputs = processor(images=image, text=concepts, return_tensors="pt", padding=True).to(device)

with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits_per_image
    probs = logits.softmax(dim=1)

# ======================
# Step 5: Display Top Predictions
# ======================
top_probs, top_labels = probs[0].topk(5)

print("\n🩺 Top Concepts Detected:")
for i in range(top_labels.shape[0]):
    label = concepts[top_labels[i]]
    score = top_probs[i].item() * 100
    print(f"🔹 {label}: {score:.2f}%")

# ======================
# Step 6: Report Highest Confidence Label
# ======================
most_likely_index = top_labels[0].item()
most_likely_label = concepts[most_likely_index]
most_likely_confidence = top_probs[0].item() * 100

print(f"\n✅ The most likely condition is: *{most_likely_label}* with a confidence of {most_likely_confidence:.2f}%.")

# ======================
# Step 7: Skin Enzyme Conclusion (If in List)
# ======================
if "skin enzyme" in concepts:
    enzyme_index = concepts.index("skin enzyme")
    enzyme_conf = probs[0, enzyme_index].item() * 100

    if enzyme_conf > 50:
        enzyme_msg = f"🧪 The image is likely related to *skin enzyme* activity (Confidence: {enzyme_conf:.2f}%)."
    else:
        enzyme_msg = f"❌ The image is not likely related to *skin enzyme* activity (Confidence: {enzyme_conf:.2f}%)."

    print("\n" + enzyme_msg)